<a href="https://colab.research.google.com/github/MagdyTarek18/Week1-Repo/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MagdyTarek18/Week1-Repo/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — Growing vs declining content

The paper reports that growing pages are generally younger and longer than declining pages.

**Methodology question:**  
The growing and declining groups come from recent impression change. I would ask whether this comparison is meant only as an observed association or whether it is being used to imply future performance.

The comparison is useful, but because it is observational, it does not prove that making a page longer or younger causes growth.

### Finding 2 — Logistic Regression for growth

The paper reports a Logistic Regression model with about 71% holdout accuracy for separating growing and declining pages.

**Methodology question:**  
Was the 80/20 holdout split grouped by client or time, or was it a random row split?

If pages from the same client appear in both training and testing, the score may partly reflect client-specific patterns. A grouped or time-aware split would give stronger evidence about generalization.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

paper_audit = pd.DataFrame({
    "finding": [
        "Growing pages are younger and longer",
        "Logistic Regression reports ~71% holdout accuracy"
    ],
    "methodology_question": [
        "Does the observational comparison support association only, rather than causation?",
        "Was the holdout split grouped by client or time?"
    ]
})

display(paper_audit)

,finding,methodology_question
0,Growing pages are younger and longer,Does the observational comparison support asso...
1,Logistic Regression reports ~71% holdout accuracy,Was the holdout split grouped by client or time?


## 2. My model under an honest split

I re-run the same Logistic Regression model from Week 5.

I compare two validation designs:

- **Before:** random train/test split
- **After:** grouped split by `client_id`

The grouped split is more honest because the same client cannot appear in both training and testing.

I compare the two using the same features and metrics.

The grouped split improves protection against client leakage, but it does not solve every possible time-window issue.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score




cwd = Path.cwd()

possible_roots = [
    cwd,
    cwd.parent,
    cwd.parent.parent
]

REPO_ROOT = cwd

for root in possible_roots:
    if (root / "data/raw/content_refresh_anonymized.csv").exists():
        REPO_ROOT = root
        break

repo_data = REPO_ROOT / "data/raw/content_refresh_anonymized.csv"
uploaded_data = Path("content_refresh_anonymized.csv")

if repo_data.exists():
    DATA_PATH = repo_data
elif uploaded_data.exists():
    DATA_PATH = uploaded_data
else:
    raise FileNotFoundError("Could not find content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)




df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

FEATURES = [
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr"
]

X = df[FEATURES].copy()
y = df["is_declining_label"].copy()


X["avg_position"] = X["avg_position"].replace(0, np.nan)




def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])


def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)
    k = min(k, len(order))

    return labels[order[:k]].mean()




X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = make_model()
random_model.fit(X_train_r, y_train_r)

random_scores = random_model.predict_proba(X_test_r)[:, 1]




groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train_g = X.iloc[train_idx]
X_test_g = X.iloc[test_idx]

y_train_g = y.iloc[train_idx]
y_test_g = y.iloc[test_idx]

grouped_model = make_model()
grouped_model.fit(X_train_g, y_train_g)

grouped_scores = grouped_model.predict_proba(X_test_g)[:, 1]




train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

assert train_clients.isdisjoint(test_clients)

print("PASS: grouped split has no client overlap.")



results = pd.DataFrame({
    "split": [
        "Random split",
        "Grouped by client"
    ],
    "test_rows": [
        len(y_test_r),
        len(y_test_g)
    ],
    "base_rate": [
        y_test_r.mean(),
        y_test_g.mean()
    ],
    "precision@10": [
        precision_at_k(random_scores, y_test_r, 10),
        precision_at_k(grouped_scores, y_test_g, 10)
    ],
    "precision@50": [
        precision_at_k(random_scores, y_test_r, 50),
        precision_at_k(grouped_scores, y_test_g, 50)
    ],
    "roc_auc": [
        roc_auc_score(y_test_r, random_scores),
        roc_auc_score(y_test_g, grouped_scores)
    ]
})

display(results.round(3))

PASS: grouped split has no client overlap.


,split,test_rows,base_rate,precision@10,precision@50,roc_auc
0,Random split,6000,0.542,0.5,0.44,0.556
1,Grouped by client,6163,0.511,0.5,0.66,0.517


### Error examples

I also inspect real mistakes from the grouped test set.

These examples help show where the model can be confident and still be wrong.

In [ ]:
grouped_pred = (
    grouped_scores >= 0.5
).astype(int)

error_examples = X_test_g.copy()

error_examples["actual"] = y_test_g.values
error_examples["predicted"] = grouped_pred
error_examples["decline_probability"] = grouped_scores

error_examples["wrong"] = (
    error_examples["actual"] !=
    error_examples["predicted"]
)

error_examples["confidence"] = np.where(
    error_examples["predicted"] == 1,
    error_examples["decline_probability"],
    1 - error_examples["decline_probability"]
)

wrong_cases = (
    error_examples[
        error_examples["wrong"]
    ]
    .sort_values(
        "confidence",
        ascending=False
    )
    .head(3)
)

print("Three confident wrong predictions:")
display(wrong_cases)

Three confident wrong predictions:


,impressions_90d,days_since_last_update,avg_position,ctr,actual,predicted,decline_probability,wrong,confidence
26844,509252,20,2.5,0.15,1,0,0.122927,True,0.877073
21819,463103,20,2.3,0.41,1,0,0.144336,True,0.855664
6653,517715,104,4.2,0.14,1,0,0.170695,True,0.829305


In [ ]:
These errors show that the measured signals do not perfectly separate declining and non-declining content.

The model should therefore be treated as **decision-support**, not as a guaranteed prediction of what will happen to a page.

## 3. Leakage audit

My target is:

`is_declining_label = 1` when `trend_direction == "down"`.

Therefore:

- `trend_direction` is a direct label source and cannot be a feature.
- `trend_pct` is also a label source and cannot be a feature.
- `client_id` is used only for splitting, not as a model feature.

My final Week-5 features were:

- `impressions_90d`
- `days_since_last_update`
- `avg_position`
- `ctr`

There is no direct label-derived feature in this list.

However, the 90-day search metrics overlap the recent time period used to construct the decline label.

This means the model is safer to describe as measuring associations with the current decline label, not as a clean future-forecasting model.

A true future prediction would require features from an earlier window and a label from a strictly later window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

FORBIDDEN = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id"
}

direct_leakage = set(FEATURES).intersection(FORBIDDEN)

print("Direct forbidden features found:", direct_leakage)

assert len(direct_leakage) == 0

print("PASS: no direct label-derived or ID features are used.")




leakage_audit = pd.DataFrame({
    "feature": FEATURES,

    "direct_label_leakage": [
        "No",
        "No",
        "No",
        "No"
    ],

    "time_window_concern": [
        "Yes - 90d window overlaps label period",
        "Low",
        "Yes - measured during overlapping period",
        "Yes - measured during overlapping period"
    ],

    "safe_interpretation": [
        "Descriptive / decision-support",
        "Safer metadata signal",
        "Descriptive / decision-support",
        "Descriptive / decision-support"
    ]
})

display(leakage_audit)

Direct forbidden features found: set()
PASS: no direct label-derived or ID features are used.


,feature,direct_label_leakage,time_window_concern,safe_interpretation
0,impressions_90d,No,Yes - 90d window overlaps label period,Descriptive / decision-support
1,days_since_last_update,No,Low,Safer metadata signal
2,avg_position,No,Yes - measured during overlapping period,Descriptive / decision-support
3,ctr,No,Yes - measured during overlapping period,Descriptive / decision-support


In [ ]:
X_leaky = df[
    FEATURES + ["trend_pct"]
].copy()

X_leaky["avg_position"] = (
    X_leaky["avg_position"]
    .replace(0, np.nan)
)

X_train_leaky = X_leaky.iloc[train_idx]
X_test_leaky = X_leaky.iloc[test_idx]

leaky_model = make_model()

leaky_model.fit(
    X_train_leaky,
    y_train_g
)

leaky_scores = leaky_model.predict_proba(
    X_test_leaky
)[:, 1]

leakage_comparison = pd.DataFrame({
    "model": [
        "Honest feature set",
        "With leaked trend_pct"
    ],

    "roc_auc": [
        roc_auc_score(
            y_test_g,
            grouped_scores
        ),

        roc_auc_score(
            y_test_g,
            leaky_scores
        )
    ]
})

display(leakage_comparison.round(3))

print(
    "trend_pct is removed after this test because "
    "it is used to create the target."
)

,model,roc_auc
0,Honest feature set,0.517
1,With leaked trend_pct,0.950


trend_pct is removed after this test because it is used to create the target.


The leaked version is shown only as a diagnostic.

`trend_pct` is not kept in the final model because it contributes directly to the definition of the decline label.

The honest result is the model that excludes this feature.

## 4. Claim rewrite

### Too strong

> "The model predicts which pages will decline."

### Safer claim

> "In this dataset, the Logistic Regression model shows a measured directional relationship between the selected content signals and the existing decline label."

Because some feature windows overlap the label period, I treat the result as **descriptive decision-support**, not proof of future prediction or causation.

The grouped-client result gives stronger evidence that the measured pattern can transfer to unseen clients, but a strict past-to-future time split would be needed before making a future-performance claim.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
claim_check = {
    "uses_future_prediction_language": False,
    "uses_causal_language": False,
    "uses_measured_language": True,
    "uses_directional_language": True,
    "uses_decision_support_language": True
}

display(pd.Series(claim_check, name="claim_check"))

,claim_check
uses_future_prediction_language,False
uses_causal_language,False
uses_measured_language,True
uses_directional_language,True
uses_decision_support_language,True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.